# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object
metadata = dataset.metadata

print(f"Name: {metadata.name}\nDescription: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This dataset includes structured tables:
- Table 1: Clinical features
- Table 2: Hormone levels, imaging, ultrasound
- Table 3: Genetic variants

We start by listing all available record sets and their field `@id`s.

In [ ]:
# List all record sets and their fields
record_sets_metadata = dataset._metadata_dict.get('recordSet', [])

record_sets_info = {}
for rs in record_sets_metadata:
    rs_id = rs.get('@id')
    rs_name = rs.get('name', 'N/A')
    fields = rs.get('field', [])
    field_ids = []
    if isinstance(fields, dict):
        fields = [fields]
    for fld in fields:
        field_ids.append(fld.get('@id'))
    record_sets_info[rs_id] = {'name': rs_name, 'fields': field_ids}

print("Record Sets and their Fields (@id):")
for rs_id, rs_data in record_sets_info.items():
    print(f"- Record Set @id: {rs_id} | Name: {rs_data['name']}")
    for fid in rs_data['fields']:
        print(f"   - Field @id: {fid}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We use the record set and field `@id`s from the overview above.

By referencing the unique `@id` for each entity, extraction is robust.

In [ ]:
# Prepare a list of record_set @ids from previous cell
record_set_ids = list(record_sets_info.keys())

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"--- Columns for record set @id '{record_set_id}':")
    print(df.columns.tolist())
    print(df.head(3))
    print()
# For subsequent analysis, select the first record set as an example
example_record_set_id = record_set_ids[0]
example_df = dataframes[example_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field (e.g., 'age at diagnosis') by its `@id` and filter, normalize, and group as examples.

In [ ]:
# Find a numeric field in the example record set
rs_fields = record_sets_info[example_record_set_id]['fields']
numeric_field_id = None

# Identify age or hormone level fields (commonly numeric)
for col in example_df.columns:
    if 'age' in col.lower() or 'level' in col.lower():
        numeric_field_id = col
        break

if numeric_field_id:
    print(f"Using numeric field '{numeric_field_id}' for EDA.")
else:
    numeric_field_id = example_df.select_dtypes(include='number').columns[0] if not example_df.empty else None
    print(f"Using fallback numeric field: {numeric_field_id}")

# Filtering threshold
threshold = 10
if not example_df.empty and numeric_field_id:
    filtered_df = example_df[example_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Identify possible grouping field (e.g., presence of a clinical feature)
    group_field_candidates = [col for col in example_df.columns if col != numeric_field_id and example_df[col].dtype == 'object']
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field and group_field in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        )
        print(f"Grouped data by '{group_field}':")
        print(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and relationship with grouping field if available.

In [ ]:
if not example_df.empty and numeric_field_id:
    plt.figure(figsize=(6,4))
    example_df[numeric_field_id].hist(bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        grouped_df = example_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(6,4))
        plt.bar(grouped_df[group_field], grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset using the `mlcroissant` library and referenced all entities by their `@id`s for reproducibility.

- We extracted metadata and reviewed available record sets and fields.
- We loaded tables into DataFrames using record set `@id`s, and examined their columns.
- We selected a numeric field (by its `@id`) for exploratory analysis, filtered and normalized values, and visualized distributions.

This structured approach facilitates robust scientific data exploration and reproducible research using standardized FAIR datasets.